[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/06_Relation_Extraction.ipynb)

#  关系抽取 —— 综合指南

##  概述

本 Notebook 是使用 Semantica 的关系抽取模块抽取实体之间的关系并构建 RDF 三元组的**综合指南**。你将学会识别关联、抽取结构化的三元组,并为知识图谱准备数据。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/semantic_extract/)

###  学习目标

学完本 Notebook 后,你将能够:

-  使用 `RelationExtractor` 抽取关系
-  理解不同的抽取方法(pattern、dependency、co-occurrence、HuggingFace、LLM)
-  配置抽取参数以获得最佳效果
-  使用 `TripletExtractor` 抽取 RDF 三元组
-  使用 `RDFSerializer` 将三元组序列化为 RDF 格式
-  构建完整的 实体 → 关系 → 三元组 管线

###  你将学到什么

| 组件 | 用途 | 何时使用 |
|-----------|---------|-------------|
| `RelationExtractor` | 抽取实体关系 | 寻找关联 |
| `TripletExtractor` | 抽取 RDF 三元组 | 构建知识图谱 |
| `RDFSerializer` | 序列化为 RDF 格式 | 数据导出 |

---

##  安装

从 PyPI 安装 Semantica:

```bash
pip install semantica
# 或安装全部可选依赖:
pip install semantica[all]
```

---

In [ ]:
!pip install semantica


##  步骤 1:基础关系抽取

让我们从使用 `RelationExtractor` 抽取实体之间的关系开始。

### 什么是 RelationExtractor?

`RelationExtractor` 用于识别实体之间的关系:
- 发现 "founded_by"、"located_in"、"works_for" 之类的关联
- 返回包含主语、谓语、宾语的 Relation 对象
- 支持多种抽取方法
- 为每个关系提供置信度分数

### 理解关系

一个关系由三部分组成:
- **主语(Subject)**:源实体(例如 "Apple Inc.")
- **谓语(Predicate)**:关系类型(例如 "founded_by")
- **宾语(Object)**:目标实体(例如 "Steve Jobs")

In [ ]:
from semantica.semantic_extract import RelationExtractor, NERExtractor

# Initialize extractors
ner_extractor = NERExtractor()
relation_extractor = RelationExtractor()

# Sample text with clear relationships
text = """
Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976.
The company is headquartered in Cupertino, California. Tim Cook is the current CEO
of Apple Inc. and took over from Steve Jobs in August 2011.
"""

# First, extract entities
entities = ner_extractor.extract(text)
print(f" Extracted {len(entities)} entities\n")

# Then, extract relationships
relationships = relation_extractor.extract(text, entities)

print(f" Extracted {len(relationships)} relationships:\n")
print("=" * 80)

for i, rel in enumerate(relationships, 1):
    # Handle both dict and object formats
    source = rel.get('source', rel.get('subject', '')) if isinstance(rel, dict) else getattr(rel, 'subject', '')
    target = rel.get('target', rel.get('object', '')) if isinstance(rel, dict) else getattr(rel, 'object', '')
    rel_type = rel.get('type', rel.get('predicate', 'related_to')) if isinstance(rel, dict) else getattr(rel, 'predicate', 'related_to')
    confidence = rel.get('confidence', 1.0) if isinstance(rel, dict) else getattr(rel, 'confidence', 1.0)
    
    # Get source and target text
    if isinstance(source, dict):
        source_text = source.get('text', source.get('entity', str(source)))
    else:
        source_text = getattr(source, 'text', str(source))
    
    if isinstance(target, dict):
        target_text = target.get('text', target.get('entity', str(target)))
    else:
        target_text = getattr(target, 'text', str(target))
    
    print(f"{i:2d}. {source_text:20s} --[{rel_type:15s}]--> {target_text:20s} (conf: {confidence:.2f})")

print("=" * 80)

###  理解关系对象

每个被抽取的关系包含:

| 属性 | 描述 | 示例 |
|-----------|-------------|----------|
| `subject` | 源实体 | Entity("Apple Inc.") |
| `predicate` | 关系类型 | "founded_by" |
| `object` | 目标实体 | Entity("Steve Jobs") |
| `confidence` | 抽取置信度(0-1) | 0.85 |
| `context` | 周围文本 | "Apple Inc. was founded by Steve Jobs" |
| `metadata` | 附加信息 | {"method": "pattern"} |

###  常见关系类型

- **founded_by**:组织由某人创立
- **located_in**:实体位于某地
- **works_for**:某人在某组织工作
- **born_in**:某人出生于某地
- **part_of**:实体是另一实体的一部分
- **related_to**:通用关系

## ️ 步骤 2:不同的抽取方法

Semantica 支持多种关系抽取方法:

### 方法比较

| 方法 | 速度 | 准确率 | 适用场景 | 依赖 |
|--------|-------|----------|----------|----------|
| **pattern** |  | ⭐⭐⭐ | 常见关系 | 无 |
| **dependency** |  | ⭐⭐⭐⭐ | 语法关系 | spaCy |
| **cooccurrence** |  | ⭐⭐ | 基于邻近度 | 无 |
| **huggingface** |  | ⭐⭐⭐⭐⭐ | 特定领域 | HF 模型 |
| **llm** |  | ⭐⭐⭐⭐⭐ | 复杂、自定义 | API 密钥 |

In [ ]:
from semantica.semantic_extract import RelationExtractor

sample_text = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
sample_entities = ner_extractor.extract(sample_text)

print(" Comparing Relation Extraction Methods:\n")
print("=" * 80)

# Try different methods
methods_to_try = ["pattern", "dependency", "cooccurrence"]

for method_name in methods_to_try:
    try:
        print(f"\n Method: {method_name.upper()}")
        print("-" * 40)
        
        extractor = RelationExtractor(method=method_name)
        relations = extractor.extract(sample_text, sample_entities)
        
        print(f"Found {len(relations)} relations:")
        for rel in relations[:3]:  # Show first 3
            source = rel.get('source', rel.get('subject', '')) if isinstance(rel, dict) else getattr(rel, 'subject', '')
            target = rel.get('target', rel.get('object', '')) if isinstance(rel, dict) else getattr(rel, 'object', '')
            rel_type = rel.get('type', rel.get('predicate', 'related_to')) if isinstance(rel, dict) else getattr(rel, 'predicate', 'related_to')
            
            # Get text representations
            source_text = source.get('text', str(source)) if isinstance(source, dict) else getattr(source, 'text', str(source))
            target_text = target.get('text', str(target)) if isinstance(target, dict) else getattr(target, 'text', str(target))
            
            print(f"  • {source_text} --[{rel_type}]--> {target_text}")
            
    except Exception as e:
        print(f"  ️  Method '{method_name}' not available: {str(e)[:50]}")

print("\n" + "=" * 80)

##  步骤 3:带配置的高级关系抽取

`RelationExtractor` 提供了强大的配置选项:

### 关键参数:

- **`relation_types`**:要抽取的特定关系类型(例如 `["founded", "works_at"]`)
- **`bidirectional`**:抽取双向关系(默认:False)
- **`confidence_threshold`**:最低置信度分数(0.0-1.0,默认:0.6)
- **`max_distance`**:实体之间的最大 Token 距离(默认:50)

In [ ]:
# Create extractor with custom configuration
advanced_extractor = RelationExtractor(
    relation_types=["founded_by", "located_in", "works_for"],  # Only extract these types
    confidence_threshold=0.7,  # Higher threshold for quality
    bidirectional=False,  # One-way relations only
    max_distance=50  # Max 50 tokens between entities
)

# Sample texts
texts = [
    "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico.",
    "Satya Nadella works for Microsoft as the CEO.",
    "Google is located in Mountain View, California.",
    "Amazon was founded by Jeff Bezos in Seattle, Washington."
]

print(" Advanced Relation Extraction:\n")
print("=" * 80)

for i, text in enumerate(texts, 1):
    entities = ner_extractor.extract(text)
    relations = advanced_extractor.extract(text, entities)
    
    print(f"\n Text {i}: {text}")
    print(f"   Relations found: {len(relations)}")
    
    for rel in relations:
        source = rel.get('source', rel.get('subject', '')) if isinstance(rel, dict) else getattr(rel, 'subject', '')
        target = rel.get('target', rel.get('object', '')) if isinstance(rel, dict) else getattr(rel, 'object', '')
        rel_type = rel.get('type', rel.get('predicate', 'related_to')) if isinstance(rel, dict) else getattr(rel, 'predicate', 'related_to')
        confidence = rel.get('confidence', 1.0) if isinstance(rel, dict) else getattr(rel, 'confidence', 1.0)
        
        source_text = source.get('text', str(source)) if isinstance(source, dict) else getattr(source, 'text', str(source))
        target_text = target.get('text', str(target)) if isinstance(target, dict) else getattr(target, 'text', str(target))
        
        print(f"     • {source_text} --[{rel_type}]--> {target_text} (conf: {confidence:.2f})")

print("\n" + "=" * 80)

## ️ 步骤 4:关系分类

按谓语类型对已抽取的关系进行分组和分类。

In [ ]:
# Extract relations from all texts
all_relations = []
for text in texts:
    entities = ner_extractor.extract(text)
    relations = advanced_extractor.extract(text, entities)
    all_relations.extend(relations)

# Classify relations
classified_relations = advanced_extractor.classify_relations(all_relations)

print("️  Relation Classification:\n")
print("=" * 80)

for rel_type, rel_list in sorted(classified_relations.items()):
    print(f"\n{rel_type.upper()} ({len(rel_list)} relations):")
    print("-" * 40)
    
    for rel in rel_list:
        source = rel.get('source', rel.get('subject', '')) if isinstance(rel, dict) else getattr(rel, 'subject', '')
        target = rel.get('target', rel.get('object', '')) if isinstance(rel, dict) else getattr(rel, 'object', '')
        
        source_text = source.get('text', str(source)) if isinstance(source, dict) else getattr(source, 'text', str(source))
        target_text = target.get('text', str(target)) if isinstance(target, dict) else getattr(target, 'text', str(target))
        
        print(f"  • {source_text} → {target_text}")

print("\n" + "=" * 80)

##  步骤 5:三元组抽取

使用 `TripletExtractor` 抽取 RDF 三元组。三元组是知识图谱的基础。

### 什么是 RDF 三元组?

RDF(资源描述框架,Resource Description Framework)三元组是由三部分组成的陈述:
- **主语(Subject)**:我们在谈论的对象
- **谓语(Predicate)**:属性或关系
- **宾语(Object)**:值或目标

示例:`(Apple Inc., founded_by, Steve Jobs)`

### 为什么使用三元组?

- 知识表示的**标准化格式**
- 与 RDF 数据库和语义网**兼容**
- 可使用 SPARQL **查询**
- 跨系统**可互操作**

In [ ]:
from semantica.semantic_extract import TripletExtractor

# Initialize triplet extractor
triplet_extractor = TripletExtractor(
    include_temporal=True,  # Include temporal information
    include_provenance=True  # Track source sentences
)

# Sample text
triplet_text = """
Apple Inc. was founded by Steve Jobs in 1976. The company is based in Cupertino, California.
Tim Cook became CEO in 2011. Apple develops the iPhone and MacBook products.
"""

# Extract triplets
triplets = triplet_extractor.extract_triplets(triplet_text)

print(f" Extracted {len(triplets)} RDF Triplets:\n")
print("=" * 80)

for i, triplet in enumerate(triplets, 1):
    subject = triplet.get('subject', '') if isinstance(triplet, dict) else triplet.subject
    predicate = triplet.get('predicate', '') if isinstance(triplet, dict) else triplet.predicate
    obj = triplet.get('object', '') if isinstance(triplet, dict) else triplet.object
    confidence = triplet.get('confidence', 1.0) if isinstance(triplet, dict) else getattr(triplet, 'confidence', 1.0)
    
    print(f"{i:2d}. ({subject}, {predicate}, {obj})")
    print(f"    Confidence: {confidence:.2f}")
    
    # Show temporal info if available
    metadata = triplet.get('metadata', {}) if isinstance(triplet, dict) else getattr(triplet, 'metadata', {})
    if metadata.get('temporal'):
        print(f"    Temporal: {metadata['temporal']}")
    print()

print("=" * 80)

##  步骤 6:RDF 序列化

使用 `RDFSerializer` 将三元组序列化为各种 RDF 格式。

### 支持的格式:

| 格式 | 扩展名 | 适用场景 |
|--------|-----------|----------|
| **Turtle** | .ttl | 人类可读、紧凑 |
| **N-Triples** | .nt | 简单、按行分隔 |
| **JSON-LD** | .jsonld | 对 Web 友好、基于 JSON |
| **RDF/XML** | .rdf | 基于 XML、较为冗长 |

In [ ]:
from semantica.semantic_extract import RDFSerializer

# Initialize serializer
serializer = RDFSerializer()

print(" RDF Serialization Examples:\n")
print("=" * 80)

# Serialize to different formats
formats = ["turtle", "ntriples", "jsonld"]

for fmt in formats:
    print(f"\n {fmt.upper()} Format:")
    print("-" * 40)
    
    try:
        serialized = serializer.serialize_to_rdf(triplets[:3], format=fmt)  # Show first 3
        
        # Show preview (first 300 chars)
        preview = serialized[:300] + "..." if len(serialized) > 300 else serialized
        print(preview)
        
    except Exception as e:
        print(f"Error: {str(e)[:50]}")

print("\n" + "=" * 80)

##  步骤 7:完整的抽取管线

让我们构建一条完整的管线:**实体 → 关系 → 三元组**

这演示了构建知识图谱的完整工作流。

In [ ]:
def extract_knowledge(text):
    """
    Complete knowledge extraction pipeline.
    
    Args:
        text: Input text
        
    Returns:
        dict: Extracted entities, relations, and triplets
    """
    # Step 1: Extract entities
    entities = ner_extractor.extract(text)
    
    # Step 2: Extract relations
    relations = relation_extractor.extract(text, entities)
    
    # Step 3: Extract triplets
    triplets = triplet_extractor.extract_triplets(text, entities=entities, relationships=relations)
    
    return {
        'entities': entities,
        'relations': relations,
        'triplets': triplets
    }

# Sample knowledge-rich text
knowledge_text = """
Tesla Inc. was founded by Elon Musk, JB Straubel, Martin Eberhard, Marc Tarpenning, 
and Ian Wright in 2003. The company is headquartered in Austin, Texas. Tesla produces 
electric vehicles including the Model S, Model 3, Model X, and Model Y. Elon Musk serves 
as CEO and has been instrumental in the company's growth.
"""

print(" Complete Extraction Pipeline:\n")
print("=" * 80)

# Run pipeline
result = extract_knowledge(knowledge_text)

print(f"\n Extraction Results:")
print("-" * 40)
print(f"Entities extracted: {len(result['entities'])}")
print(f"Relations extracted: {len(result['relations'])}")
print(f"Triplets extracted: {len(result['triplets'])}")

print(f"\n Sample Triplets:")
for i, triplet in enumerate(result['triplets'][:5], 1):
    subject = triplet.get('subject', '') if isinstance(triplet, dict) else triplet.subject
    predicate = triplet.get('predicate', '') if isinstance(triplet, dict) else triplet.predicate
    obj = triplet.get('object', '') if isinstance(triplet, dict) else triplet.object
    print(f"  {i}. ({subject}, {predicate}, {obj})")

print("\n" + "=" * 80)

##  步骤 8:最佳实践与技巧

###  选择合适的方法

1. 常见关系**从基于 pattern 的方法开始**
2. 追求语法准确性时**使用依存句法分析**
3. 探索性分析**尝试共现(co-occurrence)**
4. 复杂的、特定领域的关系**考虑 LLM**

### ️ 优化抽取

- **设置置信度阈值**(生产环境 0.6-0.7)
- **指定 relation_types** 以聚焦抽取
- 根据文本结构**调整 max_distance**

###  需避免的常见陷阱

-  **不要**跳过实体抽取(关系依赖于实体)
-  **不要**使用过低的置信度阈值
-  **不要**忘记将三元组序列化以便存储

###  各个组件的适用场景

| 使用场景 | 推荐组件 |
|----------|----------------------|
| 发现实体之间的关联 | `RelationExtractor` |
| 构建知识图谱 | `TripletExtractor` |
| 导出为 RDF | `RDFSerializer` |

###  性能技巧

1. **只抽取一次实体**,在关系和三元组中复用
2. 将多个文档**批量处理**
3. **缓存抽取器**,而不是重复创建
4. 通过置信度阈值**尽早过滤**

##  总结

### 你学到了什么

在本 Notebook 中,你学会了如何:

 使用 `RelationExtractor` **抽取关系**  
 **比较抽取方法**(pattern、dependency、co-occurrence、HuggingFace、LLM)  
 **配置抽取参数**以获得最佳效果  
 使用 `TripletExtractor` **抽取 RDF 三元组**  
 使用 `RDFSerializer` **序列化为 RDF 格式**  
 **构建从实体到三元组的完整管线**  

### 关键要点

1. **关系连接实体**:它们构成了知识图谱的骨干
2. **有多种方法可用**:根据准确率与速度的需求选择
3. **配置很强大**:针对你的领域调整参数
4. **三元组是标准化的**:使用 RDF 以实现互操作性
5. **管线很高效**:依次抽取 实体 → 关系 → 三元组

### 下一步

 **下一个 Notebook**:[07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb)  
学习如何根据抽取出的三元组构建完整的知识图谱!

 **延伸阅读**:
- [Semantic Extract API 参考](https://semantica.readthedocs.io/reference/semantic_extract/)
- [知识图谱模块](https://semantica.readthedocs.io/reference/kg/)
- [高级图分析](../advanced/02_Advanced_Graph_Analytics.ipynb)

---

**有问题?**请查看我们的 [GitHub 仓库](https://github.com/semantica-agi/semantica)或[文档](https://semantica.readthedocs.io)。